# Video branch, part 2: CLEANED screen text

Runtime → Change runtime type → **T4 GPU**, then run all cells in order (about 30–40 min, almost all Qwen).

Same 4 conditions as the raw video run, but the OCR text is first cleaned with **frozen, rule-based filters** (no LLM): credits, menu bars, status bars and garbled fragments are dropped, slide sentences / code / program output are kept, near-duplicates removed, capped at 2000 characters. The Qwen prompt, model, revision and settings are **identical** to the raw run; only the screen text changes.

Needs (already on Drive from notebook 06): `outputs/video/ocr_lines.csv`, `outputs/video/alignment.csv`, `outputs/step2/conditions.csv`.

Writes (new files only; nothing existing is overwritten):
`outputs/video/screen_text_clean.csv`, `conditions_video_clean.csv`, `clean_line_log.csv`, `summaries_video_clean.csv`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/hinglish_asr_sum'
%cd {ROOT}
!pip install -q wordfreq==3.1.1 bitsandbytes accelerate
import os
for f in ['outputs/video/ocr_lines.csv', 'outputs/video/alignment.csv', 'outputs/step2/conditions.csv']:
    print(f, 'OK' if os.path.exists(f) else 'MISSING')

In [ ]:
%%writefile clean_screen.py
"""Rule-based cleaning of OCR screen text (frozen rules; no LLM, deterministic).

Input : outputs/video/ocr_lines.csv   (window_id, t, text, conf)  - raw Tesseract lines
Output: outputs/video/screen_text_clean.csv   (window_id, n_lines, screen_text, chars)
        outputs/video/conditions_video_clean.csv (same 4 conditions as the raw run, *_clean names)
        outputs/video/clean_line_log.csv  (every raw line with keep/drop and the reason - for the paper)

    python clean_screen.py --root /content/drive/MyDrive/hinglish_asr_sum

Rules, applied in this order to each OCR line:
  1. strip bullet / OCR-symbol prefixes (», >, e, @, ®, ©, •, -, *, =)
  2. DROP credits/boilerplate: project credits, URLs, funding lines, workshop/certificate text
  3. DROP status bars / window chrome: '|', Ln/Col counters, Page n/m, Slide n/m, clock times,
     INSRT/STD, Tab Width, recorder user names
  4. DROP menu bars: >= 60% of the words are standard menu words (File, Edit, View, ...)
  5. KEEP code / commands / program output: line has code characters or tokens (; ( ) # $ ./ % == gcc printf ...)
  6. otherwise KEEP only sentence-like text: >= 4 words and >= 75% of alphabetic words are real English
     words (wordfreq zipf >= 2.0) - this removes garbled fragments and single menu items
  7. merge consecutive kept lines with the same timestamp when the first does not end a sentence and
     the next starts in lower case (re-joins slide bullets split by OCR)
  8. DROP near-duplicates: a line whose word set is >= 80% covered by an earlier kept line
  9. cap at 2000 characters, cut at a line boundary (time order kept, like the raw condition)
"""
import argparse
import os
import re

import pandas as pd
from wordfreq import zipf_frequency

CAP = 2000
BULLET = re.compile(r"^[\s»>«@®©•*=+\-–—~|e]{1,4}\s+(?=\S)|^[»>«@®©•]+")
CREDITS = re.compile(r"national mission|\bict\b|sakshat|talk to a teacher|desicrew|spoken[ -]?tutorial|script and "
                     r"narration|contributed by|it for change|https?:|www\.|\.org\b|\.ac\.in|mhrd|workshop|certificate|"
                     r"acknowledg|government of india|iit bombay|nmeict|what_is_a|bandwidth|online test|"
                     r"write to contact|pvt\.? ?ltd", re.I)
STATUS = re.compile(r"\||\bln\s*\d|\bcol\s*\d|page\s*\d+\s*/\s*\d+|slide\s*\d+\s*/\s*\d+|\d{1,2}:\d{2}\s*[ap]m|"
                    r"\binsrt\b|\bstd\b|tab width|\bins\b$|ttt@ttt|recordmydesktop|applications places", re.I)
MENU = set("file edit view insert format tools table window help slide show search documents go message "
           "applications places system bookmarks history project run terminal".split())
CODE = re.compile(r"[;{}#$]|\(\s*\)|\w\(|\./\w|%[ds]|==|\bgcc\b|\bprintf\b|\bscanf\b|\bint\b|\bchar\b|"
                  r"\bvoid\b|\breturn\b|\.c\b|\.h\b|->|\bargc\b|\bargv\b")
WORD = re.compile(r"[A-Za-z][A-Za-z'\-]*")


def is_word(w):
    return zipf_frequency(w.lower().strip("'-"), "en") >= 2.0


def classify(text):
    t = BULLET.sub("", text).strip()
    words = WORD.findall(t)
    if not t:
        return t, "drop:empty"
    if CREDITS.search(t):
        return t, "drop:credits"
    if STATUS.search(t):
        return t, "drop:status_bar"
    if words and sum(w.lower() in MENU for w in words) / len(words) >= 0.6:
        return t, "drop:menu_bar"
    if CODE.search(t) and len(t) >= 6:
        return t, "keep:code"
    if len(words) >= 4 and sum(is_word(w) for w in words) / len(words) >= 0.75:
        return t, "keep:sentence"
    return t, "drop:fragment"


def clean_window(g):
    """g: DataFrame of one window's raw lines with columns t, text (any order). Returns (text, log rows)."""
    log, kept = [], []
    for t, raw in zip(g.t, g.text):
        txt, why = classify(str(raw))
        log.append(dict(t=t, raw=raw, cleaned=txt, decision=why))
        if why.startswith("keep"):
            kept.append([t, txt])
    merged = []                                                     # rule 7
    for t, txt in kept:
        if merged and merged[-1][0] == t and not re.search(r"[.!?:;]$", merged[-1][1]) and txt[:1].islower():
            merged[-1][1] += " " + txt
        else:
            merged.append([t, txt])
    final, seen = [], []                                            # rule 8
    for t, txt in merged:
        ws = {w.lower() for w in WORD.findall(txt)}
        if ws and any(len(ws & s) / len(ws) >= 0.8 for s in seen):
            continue
        seen.append(ws)
        final.append(f"[{int(t // 60):02d}:{int(t % 60):02d}] {txt}")
    out = "\n".join(final)
    if len(out) > CAP:                                              # rule 9
        out = out[:CAP].rsplit("\n", 1)[0]
    return out, len(final), log


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True)
    A = ap.parse_args()
    OUT = f"{A.root}/outputs/video"
    ocr = pd.read_csv(f"{OUT}/ocr_lines.csv", keep_default_na=False)
    ocr["t"] = ocr.t.astype(float)
    cond = pd.read_csv(f"{A.root}/outputs/step2/conditions.csv", keep_default_na=False)
    wins = sorted(pd.read_csv(f"{OUT}/alignment.csv").window_id)
    st, rows, logs = [], [], []
    for w in wins:
        g = ocr[ocr.window_id == w].sort_values("t", kind="stable")
        text, n, log = clean_window(g)
        logs += [dict(window_id=w, **r) for r in log]
        st.append(dict(window_id=w, n_lines=n, screen_text=text, chars=len(text)))
        for c in ["oracle", "asr:trelis-hinglish-preview", "asr:whisper-small"]:
            tr = cond[(cond.window_id == w) & (cond.condition == c)].text.iloc[0]
            rows.append(dict(window_id=w, condition=f"{c}+screen_clean", transcript=tr, screen_text=text))
        rows.append(dict(window_id=w, condition="screen_clean_only", transcript="", screen_text=text))
    pd.DataFrame(st).to_csv(f"{OUT}/screen_text_clean.csv", index=False)
    pd.DataFrame(rows).to_csv(f"{OUT}/conditions_video_clean.csv", index=False)
    lg = pd.DataFrame(logs)
    lg.to_csv(f"{OUT}/clean_line_log.csv", index=False)
    print(lg.decision.value_counts().to_string())
    print(pd.DataFrame(st)[["window_id", "n_lines", "chars"]].to_string(index=False))


In [ ]:
%%writefile run_qwen_video.py
"""Summarise the video conditions with the same Qwen setup as Step 2 (resumable).
    python run_qwen_video.py --root ROOT
Reads outputs/video/conditions_video.csv; writes outputs/video/summaries_video.csv.
Settings identical to Step 2 (Qwen2.5-7B-Instruct @ a09a354, nf4 4-bit, greedy, batch 4, max_new_tokens 320).
The transcript+screen prompt is the Step-2 summary_en prompt with one added sentence and a screen-text block;
the screen-only prompt describes OCR text instead of a transcript. Both are logged to prompts_log.jsonl."""
import argparse, datetime as dt, glob, hashlib, json, os
import pandas as pd, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

ap = argparse.ArgumentParser(); ap.add_argument("--root", required=True)
ap.add_argument("--set", choices=["raw", "clean"], default="raw", help="raw OCR (default) or cleaned OCR text")
A = ap.parse_args()
SFX = "" if A.set == "raw" else "_clean"
ROOT = A.root; OUT = f"{ROOT}/outputs/video"; CACHE = f"{ROOT}/cache/qwen_video"; os.makedirs(CACHE, exist_ok=True)
MODEL, REV, BATCH = "Qwen/Qwen2.5-7B-Instruct", "a09a35458c702b33eeacc393d103063234e8bc28", 4
GEN = dict(max_new_tokens=320, do_sample=False)

logged = [json.loads(l) for l in open(f"{ROOT}/logs/prompts_log.jsonl")]
base = [d["prompt"] for d in logged if d.get("component") == "summary_en" and d.get("step") == "2-templates"][0]
SYSTEM, BASE_T = base["system"], base["user_template"]
head, tail = BASE_T.split("\n\nTranscript:\n")            # tail = "<<<\n{transcript}\n>>>"
assert tail == "<<<\n{transcript}\n>>>", "unexpected Step-2 template"
SCREEN_T = (head + "\n\nAfter the transcript you also get the text visible on screen in the tutorial video, read "
            "by OCR in time order; it may contain menu names, commands and recognition errors.\n\nTranscript:\n"
            + tail + "\n\nScreen text:\n<<<\n{screen}\n>>>")
ONLY_T = ("The following is the text visible on screen in a recorded computer-software tutorial video, read by OCR "
          "in time order; it may contain menu names, commands and recognition errors. There is no transcript.\n\n"
          "Write a summary in English of 120-180 words covering the main steps, concepts and facts taught. "
          "Write plain prose: no bullet points, no title.\n\nScreen text:\n<<<\n{screen}\n>>>")

cv = pd.read_csv(f"{OUT}/conditions_video{SFX}.csv", keep_default_na=False)
def prompt(r):
    u = ONLY_T.format(screen=r.screen_text) if r.condition.endswith("_only") else \
        SCREEN_T.format(transcript=r.transcript, screen=r.screen_text)
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": u}]
key = lambda w, c: hashlib.sha256(f"{w}|{c}|video".encode()).hexdigest()[:24]
todo = [r for r in cv.itertuples() if not os.path.exists(f"{CACHE}/{key(r.window_id, r.condition)}.json")]
print(f"{len(cv) - len(todo)} cached, {len(todo)} to generate")
if todo:
    tok = AutoTokenizer.from_pretrained(MODEL, revision=REV); tok.padding_side = "left"
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(MODEL, revision=REV, quantization_config=bnb, device_map="auto",
                                                 dtype=torch.float16).eval()
    for i in range(0, len(todo), BATCH):
        b = todo[i:i + BATCH]
        enc = tok([tok.apply_chat_template(prompt(r), tokenize=False, add_generation_prompt=True) for r in b],
                  return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, **GEN, pad_token_id=tok.eos_token_id)
        for k, r in enumerate(b):
            txt = tok.decode(out[k][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()
            json.dump(dict(created=dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"), model=MODEL,
                           revision=REV, gen=str(GEN), batch=BATCH, window_id=r.window_id, condition=r.condition,
                           out_lang="en", input_tokens=int(enc["attention_mask"][k].sum()), summary=txt),
                      open(f"{CACHE}/{key(r.window_id, r.condition)}.json", "w"), ensure_ascii=False)
        print(f"done {min(i + BATCH, len(todo))}/{len(todo)}", flush=True)
    with open(f"{ROOT}/logs/prompts_log.jsonl", "a") as f:
        for comp, t in [("summary_en_screen", SCREEN_T), ("summary_en_screen_only", ONLY_T)]:
            f.write(json.dumps({"ts": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"), "step": "3-video" + SFX,
                                "component": comp, "model": MODEL, "params": dict(revision=REV, **GEN, batch=BATCH),
                                "prompt": {"system": SYSTEM, "user_template": t}}, ensure_ascii=False) + "\n")
df = pd.DataFrame([json.load(open(p)) for p in glob.glob(f"{CACHE}/*.json")])
df = df[df.condition.isin(set(cv.condition))]          # only this set's conditions
df["summary_words"] = df.summary.str.split().str.len()
df["ends_clean"] = df.summary.str.strip().str[-1].isin(list('.!?"\')'))
df.sort_values(["condition", "window_id"]).to_csv(f"{OUT}/summaries_video{SFX}.csv", index=False)
print(df.groupby("condition")[["summary_words", "input_tokens", "ends_clean"]].mean().round(1))


## Step 1: clean the screen text (CPU, ~1 min)
Prints how many raw OCR lines were kept/dropped and the cleaned length per window.

In [ ]:
!python clean_screen.py --root "$ROOT"
import pandas as pd
st = pd.read_csv(f'{ROOT}/outputs/video/screen_text_clean.csv', keep_default_na=False)
print(st.loc[st.window_id == 'W06', 'screen_text'].iloc[0])   # quick look at one window

## Step 2: Qwen summaries with the cleaned text (GPU, 100 summaries)
Safe to re-run after a disconnect: finished summaries are cached and skipped. Expect `0 cached, 100 to generate` on the first run.

In [ ]:
!python run_qwen_video.py --root "$ROOT" --set clean

## Step 3: download the results
Send me these 3 files.

In [ ]:
from google.colab import files
for f in ['summaries_video_clean.csv', 'screen_text_clean.csv', 'clean_line_log.csv']:
    files.download(f'{ROOT}/outputs/video/{f}')